In [2]:
import requests
from bs4 import BeautifulSoup

def fetch_listing_html(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    return response.text

# Paste a real Craigslist car listing URL to test
test_url = "https://www.craigslist.org/view/d/poughkeepsie-2003-ford-excursion/gUNKthbqotTAKUyZ8Dcr7y"
html = fetch_listing_html(test_url)
print(f"Fetched {len(html):,} characters")
print(html[:500])

Fetched 35,664 characters
<!DOCTYPE html>
<html>
<head>
    
	<meta charset="UTF-8">
	<meta http-equiv="X-UA-Compatible" content="IE=Edge">
	<meta name="viewport" content="width=device-width,initial-scale=1">
	<meta property="og:site_name" content="craigslist">
	<meta name="twitter:card" content="preview">
	<meta property="og:title" content="2003 ford excursion diesel for sale - Poughkeepsie, NY - craigslist">
	<meta name="description" content="2003 Ford Excursion Limited 6.0L V8 Turbo Diesel 4x4, Black with Tan leather.


In [3]:
def extract_visible_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # Strip elements that won't contain useful listing text
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    # Collapse excessive blank lines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

listing_text = extract_visible_text(html)
print(f"Extracted {len(listing_text):,} characters of visible text")
print(listing_text[:1500])

Extracted 1,867 characters of visible text
2003 ford excursion diesel for sale - Poughkeepsie, NY - craigslist
CL
hudson valley
>
for sale by dealer
>
cars+trucks
post
account
favorites
hidden
CL
hudson valley            >
cars & trucks - by dealer
...
◀  prev
▲
next ▶
favorite
favorite
hide
unhide
⚐
⚑
flag
⚑
flagged
Posted
2026-07-10 12:57
Contact Information:
print
2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!
-
$23,995
(Hyde Park/Poughkeepsie)
‹
image 1 of 15
›
120 E Dorsey Lane
2003
ford excursion
condition:
excellent
cylinders:
8 cylinders
drive:
4wd
fuel:
diesel
odometer:
151,000
paint color:
black
title status:
clean
transmission:
automatic
type:
SUV
more ads by this seller
QR Code Link to This Post
2003 Ford Excursion Limited 6.0L V8 Turbo Diesel 4x4, Black with Tan leather. This 7-Passenger Diesel SUV is *very clean* and runs and drives excellent! It has 151,000 miles and is equipped with a Clean Carfax, power windows and locks, power leather heated se

In [4]:
from pydantic import BaseModel, Field
from typing import Optional
from litellm import completion
import json

class ExtractedListing(BaseModel):
    title: str = Field(description="Full listing title, e.g. '2003 Ford Excursion Limited'")
    make: str
    model: str
    trim: Optional[str] = None
    year: int
    mileage: float = Field(description="Odometer reading in miles")
    price: float = Field(description="Asking price in dollars, numeric only")
    horsepower: Optional[float] = None
    body_type: Optional[str] = None
    fuel_type: Optional[str] = None
    transmission: Optional[str] = None
    has_accidents: Optional[bool] = None
    frame_damaged: Optional[bool] = None
    summary: str = Field(description="1-3 sentence plain description of the car's condition and features")

def extract_car_details(listing_text: str) -> ExtractedListing:
    prompt = f"""Extract structured car listing data from this text. Return ONLY valid JSON matching this schema, no explanation:

{{
  "title": "...", "make": "...", "model": "...", "trim": "...", "year": ...,
  "mileage": ..., "price": ..., "horsepower": ..., "body_type": "...",
  "fuel_type": "...", "transmission": "...", "has_accidents": ..., "frame_damaged": ...,
  "summary": "..."
}}

Use null for fields you cannot confidently determine. Do not guess has_accidents or frame_damaged unless the text explicitly mentions accident/damage history.

Listing text:
{listing_text}"""

    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    data = json.loads(raw)
    return ExtractedListing(**data)

extracted = extract_car_details(listing_text)
print(extracted.model_dump_json(indent=2))

{
  "title": "2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!",
  "make": "Ford",
  "model": "Excursion",
  "trim": "Limited",
  "year": 2003,
  "mileage": 151000.0,
  "price": 23995.0,
  "horsepower": null,
  "body_type": "SUV",
  "fuel_type": "Diesel",
  "transmission": "automatic",
  "has_accidents": false,
  "frame_damaged": false,
  "summary": "2003 Ford Excursion Limited 6.0L V8 Turbo Diesel 4x4, black with tan leather. This 7-passenger SUV is very clean and runs and drives excellent with 151,000 miles. It is equipped with a Clean Carfax, power windows and locks, power leather heated seats, captain chairs in second row, and 2 keys. The title status is clean."
}


In [5]:
def extract_car_details(listing_text: str) -> ExtractedListing:
    prompt = f"""Extract structured car listing data from this text. Return ONLY valid JSON matching this schema, no explanation:

{{
  "title": "...", "make": "...", "model": "...", "trim": "...", "year": ...,
  "mileage": ..., "price": ..., "horsepower": ..., "body_type": "...",
  "fuel_type": "...", "transmission": "...", "has_accidents": ..., "frame_damaged": ...,
  "summary": "..."
}}

For has_accidents and frame_damaged: use null unless the text EXPLICITLY states
an accident or frame damage occurred (e.g. "was in an accident", "frame damage",
"salvage title"). Do NOT infer false from phrases like "clean Carfax", "clean title",
or "no issues" — those are not the same claim as "confirmed no accidents". Only set
these to true or false when the text directly addresses accident/damage history.

Listing text:
{listing_text}"""

    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    data = json.loads(raw)
    return ExtractedListing(**data)

extracted = extract_car_details(listing_text)
print(extracted.model_dump_json(indent=2))

{
  "title": "2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!",
  "make": "Ford",
  "model": "Excursion",
  "trim": "Limited 6.0L Turbo Diesel",
  "year": 2003,
  "mileage": 151000.0,
  "price": 23995.0,
  "horsepower": null,
  "body_type": "SUV",
  "fuel_type": "diesel",
  "transmission": "automatic",
  "has_accidents": null,
  "frame_damaged": null,
  "summary": "This 7-Passenger Diesel SUV in black with tan leather is very clean and runs and drives excellent! It has 151,000 miles and is equipped with power windows and locks, power leather heated seats, captain chairs in the second row, and 2 keys. Comes with a Clean Carfax report. No hidden dealer fees. Extended warranties and financing options are available."
}


In [6]:
import sys
sys.path.append('..')
from auto_pricer.car import Car

def to_car(extracted: ExtractedListing) -> Car:
    return Car(
        title=extracted.title,
        make=extracted.make,
        model=extracted.model,
        trim=extracted.trim,
        year=extracted.year,
        mileage=extracted.mileage,
        price=extracted.price,
        horsepower=extracted.horsepower,
        body_type=extracted.body_type,
        fuel_type=extracted.fuel_type,
        transmission=extracted.transmission,
        has_accidents=extracted.has_accidents,
        frame_damaged=extracted.frame_damaged,
        summary=extracted.summary,
    )

car = to_car(extracted)
print(car)
print(f"\nSummary used for pricing:\n{car.summary}")

title='2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!' make='Ford' model='Excursion' trim='Limited 6.0L Turbo Diesel' year=2003 mileage=151000.0 price=23995.0 horsepower=None body_type='SUV' fuel_type='diesel' transmission='automatic' has_accidents=None frame_damaged=None full=None summary='This 7-Passenger Diesel SUV in black with tan leather is very clean and runs and drives excellent! It has 151,000 miles and is equipped with power windows and locks, power leather heated seats, captain chairs in the second row, and 2 keys. Comes with a Clean Carfax report. No hidden dealer fees. Extended warranties and financing options are available.' prompt=None completion=None id=None

Summary used for pricing:
This 7-Passenger Diesel SUV in black with tan leather is very clean and runs and drives excellent! It has 151,000 miles and is equipped with power windows and locks, power leather heated seats, captain chairs in the second row, and 2 keys. Comes with a Clean Carfax 

In [7]:
from auto_pricer.agents import EnsembleAgent

ensemble = EnsembleAgent()
predicted_price = ensemble.price(car.summary)

print(f"Scraped listing price (seller's asking price): ${car.price:,.0f}")
print(f"EnsembleAgent predicted price: ${predicted_price:,.0f}")
print(f"Difference: ${predicted_price - car.price:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Scraped listing price (seller's asking price): $23,995
EnsembleAgent predicted price: $17,498
Difference: $-6,498


In [8]:
from auto_pricer.agents import SpecialistAgent, FrontierAgent

specialist = SpecialistAgent()
frontier = FrontierAgent()

p_specialist = specialist.price(car.summary)
p_frontier = frontier.price(car.summary)

print(f"Asking price: ${car.price:,.0f}")
print(f"Specialist (fine-tuned, no RAG): ${p_specialist:,.0f}")
print(f"Frontier (Gemini + RAG): ${p_frontier:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Asking price: $23,995
Specialist (fine-tuned, no RAG): $13,995
Frontier (Gemini + RAG): $16,995


In [9]:
p_frontier_1 = frontier.price(car.summary)
p_frontier_2 = frontier.price(car.summary)
print(f"Frontier call 1: ${p_frontier_1:,.0f}")
print(f"Frontier call 2: ${p_frontier_2:,.0f}")

Frontier call 1: $16,495
Frontier call 2: $14,450


In [10]:
docs, prices = frontier._find_similars(car.summary)
for doc, price in zip(docs, prices):
    print(f"${price:,.0f} — {doc[:100]}")

$32,995 — Title: 2016 Lexus RX 350 AWD
Category: SUV
Make: Lexus
Description: This one-owner Lexus RX 350 is i
$32,188 — Title: 2019 Acura RDX FWD
Category: SUV
Make: Acura
Description: One owner, non-smoker vehicle in ex
$15,772 — Title: 2013 Lexus RX 350 AWD SUV
Category: SUV
Make: Lexus
Description: This well-maintained 2013 Le
$17,444 — Title: 2014 Lexus RX 350 SUV
Category: SUV
Make: Lexus
Description: This one-owner, clean Carfax RX 
$18,495 — Title: 2013 Lexus RX 350 FWD
Category: SUV
Make: Lexus
Description: This one-owner 2013 Lexus RX 350


In [12]:
import sys
sys.path.append('..')
from auto_pricer.agents import FrontierAgent

frontier = FrontierAgent()
p1 = frontier.price(car.summary)
p2 = frontier.price(car.summary)
print(f"Frontier call 1: ${p1:,.0f}")
print(f"Frontier call 2: ${p2:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Frontier call 1: $16,995
Frontier call 2: $14,995


In [16]:
import asyncio
import sys

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

def run_smoke_test():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto("https://example.com")
        title = page.title()
        browser.close()
        return title

with ThreadPoolExecutor(1) as executor:
    result = executor.submit(run_smoke_test).result()

print(result)

Example Domain


In [17]:
def fetch_fb_listing(url: str) -> str:
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, timeout=30000)
        page.wait_for_timeout(3000)  # let JS render
        html = page.content()
        browser.close()
        return html

with ThreadPoolExecutor(1) as executor:
    fb_html = executor.submit(fetch_fb_listing, "https://www.facebook.com/marketplace/item/1524895609094527/?ref=browse_tab&referral_code=marketplace_top_picks&referral_story_type=top_picks").result()

print(f"Fetched {len(fb_html):,} characters")
print(fb_html[:1000])

Fetched 1,219,719 characters
<!DOCTYPE html><html id="facebook" class="_9dls __fb-light-mode" lang="en" dir="ltr"><head><link data-default-icon="https://static.xx.fbcdn.net/rsrc.php/y1/r/ay1hV6OlegS.ico" data-badged-icon="https://static.xx.fbcdn.net/rsrc.php/yD/r/UJj0tgk-RrT.ico" rel="shortcut icon" href="https://static.xx.fbcdn.net/rsrc.php/y1/r/ay1hV6OlegS.ico"><meta name="bingbot" content="noarchive"><meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=2,shrink-to-fit=no"><meta property="al:android:app_name" content="Facebook"><meta property="al:android:package" content="com.facebook.katana"><meta property="al:android:url" content="fb://marketplace_product_details/?id=1524895609094527"><meta property="al:ios:app_name" content="Facebook"><meta property="al:ios:app_store_id" content="284882215"><meta property="al:ios:url" content="fb://marketplace_product_details/?id=1524895609094527"><meta name="apple-itunes-app" content="app-id=284882215, app-argument=fb:/

In [18]:
# Check for login-wall indicators
login_signals = ["Log in to Facebook", "You must log in", "log in or sign up", "Log In"]
found_signals = [s for s in login_signals if s.lower() in fb_html.lower()]
print(f"Login-wall signals found: {found_signals}")

# Extract visible text the same way as Craigslist
def extract_visible_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

fb_text = extract_visible_text(fb_html)
print(f"\nExtracted {len(fb_text):,} characters of visible text")
print(fb_text[:1500])

Login-wall signals found: ['Log in to Facebook', 'Log In']

Extracted 2,881 characters of visible text
2011 BMW 328i xdrive - Cars & Trucks - Florida, New York | Facebook Marketplace | Facebook
Facebook
Log In
Log In
Forgot Account?
Marketplace
Browse all
Jobs
Your account
Create new listing
Location
Florida, New York
·
Within 40 mi
Categories
Vehicles
Property Rentals
Apparel
Classifieds
Electronics
Entertainment
Family
Free Stuff
Garden & Outdoor
Hobbies
Home Goods
Home Improvement Supplies
Home Sales
Musical Instruments
Office Supplies
Pet Supplies
Sporting Goods
Toys & Games
More categories
Nearby Cities
Chester, New York
Warwick, New York
Goshen, New York
New Hampton, New York
Pine Island, New York
See more
0:03
/
0:14
2011 BMW 328i xdrive
$5,000
Listed
3 weeks ago
in
Florida, NY
Message
Message
Save
Share
Save
Share
Details
Condition
Used - Good
153xxx miles
Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to 

In [19]:
extracted_fb = extract_car_details(fb_text)
print(extracted_fb.model_dump_json(indent=2))

{
  "title": "2011 BMW 328i xdrive",
  "make": "BMW",
  "model": "328i",
  "trim": "xdrive",
  "year": 2011,
  "mileage": 153000.0,
  "price": 5000.0,
  "horsepower": null,
  "body_type": null,
  "fuel_type": null,
  "transmission": null,
  "has_accidents": null,
  "frame_damaged": null,
  "summary": "Condition Used - Good. 153xxx miles. Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to date. TPMS system fully working. 4 matching good tires. Active headlights (turn left and right with steering wheel). Heated leather seats and heated steering wheel. Radio with factory subs under the seats work perfectly. Power everything and sun/moon roof. Work done within the last year: Front lower/rearward control arms, Rear shocks, Front right CX axle, Both front wheel bearings, Brakes and sensors all around, Alignment, Oil changed every 10k with full synthetic and OEM filter. Great daily driver. Seller is looking at another car

In [20]:
car_fb = to_car(extracted_fb)
predicted_price_fb = ensemble.price(car_fb.summary)

print(f"Asking price: ${car_fb.price:,.0f}")
print(f"EnsembleAgent predicted price: ${predicted_price_fb:,.0f}")

Asking price: $5,000
EnsembleAgent predicted price: $14,872


In [21]:
def format_summary_for_pricing(extracted: ExtractedListing) -> str:
    category = extracted.body_type or "Vehicle"
    details_parts = [f"{extracted.mileage:,.0f} miles"]
    if extracted.fuel_type:
        details_parts.append(extracted.fuel_type)
    if extracted.transmission:
        details_parts.append(extracted.transmission)
    details = ", ".join(details_parts)

    return (
        f"Title: {extracted.year} {extracted.make} {extracted.model} {extracted.trim or ''}".strip() + "\n"
        f"Category: {category}\n"
        f"Make: {extracted.make}\n"
        f"Description: {extracted.summary}\n"
        f"Details: {details}."
    )

# Rebuild both test cars with the corrected summary format
car_fb.summary = format_summary_for_pricing(extracted_fb)
print(car_fb.summary)

Title: 2011 BMW 328i xdrive
Category: Vehicle
Make: BMW
Description: Condition Used - Good. 153xxx miles. Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to date. TPMS system fully working. 4 matching good tires. Active headlights (turn left and right with steering wheel). Heated leather seats and heated steering wheel. Radio with factory subs under the seats work perfectly. Power everything and sun/moon roof. Work done within the last year: Front lower/rearward control arms, Rear shocks, Front right CX axle, Both front wheel bearings, Brakes and sensors all around, Alignment, Oil changed every 10k with full synthetic and OEM filter. Great daily driver. Seller is looking at another car, which is the only reason for selling. One wheel doesn't match, seller has the matching one but it needs to be repaired.
Details: 153,000 miles.


In [22]:
predicted_price_fb_v2 = ensemble.price(car_fb.summary)
print(f"Asking price: ${car_fb.price:,.0f}")
print(f"EnsembleAgent predicted price (v1, prose summary): $14,872")
print(f"EnsembleAgent predicted price (v2, structured template): ${predicted_price_fb_v2:,.0f}")

Asking price: $5,000
EnsembleAgent predicted price (v1, prose summary): $14,872
EnsembleAgent predicted price (v2, structured template): $9,122
